# Using RXIMO to get explanations

This notebook provides a brief introduction to R-XIMO and demonstrates how it can be used to generate explanations about tradeoff information. R-XIMO is intended to make interactive methods more interpretable by helping users understand which objective functions contribute most to an obtained solution, making it easier to provide new preference information. The purpose of this notebook is to walk through the basic workflow for using R-XIMO to obtain and examine these explanations in practice.

## `ShapExplainer` with the river pollution problem

This example uses the analytical formulation from `desdeo.problem.testproblems.river_pollution_problems` instead of the precomputed CSV data.

The workflow now matches the reference-point setting more closely:

1. Define the river pollution problem and generate surrogate data by evaluating many sampled decision vectors from the analytical formulation.
2. Let the decision maker provide a reference point in the objective space.
3. Solve an achievement scalarizing function to obtain a feasible solution whose objective values are close to that reference point.
4. Generate background data around the achieved objective vector and explain how `x_1` and `x_2` contributed to the final outcome.

In [13]:
import numpy as np
import polars as pl

from desdeo.explanations import ShapExplainer, generate_biased_mean_data
from desdeo.problem.testproblems import river_pollution_problem
from desdeo.mcdm.reference_point_method import rpm_solve_solutions

problem = river_pollution_problem(five_objective_variant=True)

output_symbols = [objective.symbol for objective in problem.objectives]
# Inputs are DM-provided reference point components z_f_1 ... z_f_5
input_symbols = [f"z_{objective.symbol}" for objective in problem.objectives]

# Sample reference points in the [ideal, nadir] range of each objective
rng = np.random.default_rng(seed=1)
n_samples = 10

sampled_reference_points = {
    f"z_{objective.symbol}": rng.uniform(
        low=min(float(objective.ideal), float(objective.nadir)),
        high=max(float(objective.ideal), float(objective.nadir)),
        size=n_samples,
    )
    for objective in problem.objectives
}

# Evaluate each sampled reference point with RPM to get achieved objective values
sampled_objectives = []
for i in range(n_samples):
    rp = {
        objective.symbol: sampled_reference_points[f"z_{objective.symbol}"][i]
        for objective in problem.objectives
    }
    result = rpm_solve_solutions(problem=problem, reference_point=rp)[0]
    sampled_objectives.append({symbol: result.optimal_objectives[symbol] for symbol in output_symbols})

# Build dataset for the explainer: inputs are reference-point coordinates, outputs are achieved objectives
problem_data = pl.DataFrame(
    {
        **{input_symbol: sampled_reference_points[input_symbol] for input_symbol in input_symbols},
        **{output_symbol: [obj[output_symbol] for obj in sampled_objectives] for output_symbol in output_symbols},
    }
)

explainer = ShapExplainer(
    problem_data=problem_data,
    input_symbols=input_symbols,
    output_symbols=output_symbols,
)

problem_data.head()

c:\Users\Giomara\Documents\Projects\RXIMO_viz\.venv\Lib\site-packages\nevergrad\parametrization\_datalayers.py:107: NevergradRuntimeWarning: Bounds are 0.7 sigma away from each other at the closest, you should aim for at least 3 for better quality.
  warnings.warn(
c:\Users\Giomara\Documents\Projects\RXIMO_viz\.venv\Lib\site-packages\nevergrad\parametrization\_datalayers.py:107: NevergradRuntimeWarning: Bounds are 0.7 sigma away from each other at the closest, you should aim for at least 3 for better quality.
  warnings.warn(
c:\Users\Giomara\Documents\Projects\RXIMO_viz\.venv\Lib\site-packages\scipy\_lib\pyprima\common\preproc.py:202: UserWarning: COBYLA: Invalid RHOEND; it should be a positive number and RHOEND <= RHOBEG; it is set to 1e-06
  warn(f'{solver}: Invalid RHOEND; it should be a positive number and RHOEND <= RHOBEG; it is set to {rhoend}')
c:\Users\Giomara\Documents\Projects\RXIMO_viz\.venv\Lib\site-packages\nevergrad\parametrization\_datalayers.py:107: NevergradRuntimeWar

z_f_1,z_f_2,z_f_3,z_f_4,z_f_5,f_1,f_2,f_3,f_4,f_5
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
5.563796,3.294573,5.707618,-4.694135,0.224465,5.330166,3.207734,7.301863,-3.167959,0.27598
6.261237,3.167505,2.333335,-8.576104,0.298421,6.34,3.235212,0.321111,-3.130527,0.35
4.979214,3.044542,3.803671,-3.652149,0.207529,5.632134,3.081771,7.058205,-1.488769,0.185444
6.258353,3.315173,7.361693,-2.166174,0.091034,5.754808,3.113484,6.893064,-1.788914,0.210681
5.245812,3.028885,7.224699,-3.753868,0.293959,5.270025,3.113901,7.334034,-1.896833,0.218311
